# Continuous-time Dolinar receiver

The previous simulations approximated adaptive feedback using a finite
number of temporal stages.

Here, the receiver is instead simulated as a continuous photon-counting
measurement approximated by many short time intervals.

For binary coherent states $|\pm\alpha\rangle$, the optimal Dolinar
feedback waveform is

$$
u(t)=
\frac{(-1)^{N(t)}\alpha}
{\sqrt{1-4p(1-p)e^{-4\bar n t}}},
$$

where $N(t)$ is the number of photons detected before time $t$.

For equal priors, $p=1/2$,

$$
u(t)=
\frac{(-1)^{N(t)}\alpha}
{\sqrt{1-e^{-4\bar n t}}}.
$$

Each photon detection reverses the sign of the displacement field and
therefore reverses the current hypothesis.

For a short interval $dt$, the displaced field produces a photon-counting
rate

$$
\lambda_\pm(t)=|\pm\alpha-u(t)|^2.
$$

The probability of at least one photon detection during the interval is

$$
P(\mathrm{click})=1-e^{-\lambda_\pm(t)dt}.
$$

Taking increasingly small time intervals approximates the continuous
feedback measurement.

For equal priors the ideal feedback amplitude diverges at $t=0$.
A finite maximum displacement is therefore used in the numerical
simulation, similar to a physical implementation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from dolinar_receiver.receivers import helstrom_error

In [ ]:
def continuous_dolinar_error(
    nbar,
    shots=100_000,
    time_steps=1024,
    max_ratio=50.0,
    seed=1234
):
    rng = np.random.default_rng(seed)

    alpha = np.sqrt(nbar)
    dt = 1.0 / time_steps

    # Randomly transmit |+alpha> or |-alpha>
    sent_plus = rng.random(shots) < 0.5

    signal = np.where(
        sent_plus,
        +alpha,
        -alpha
    )

    # False = even number of detections
    # True  = odd number of detections
    odd_parity = np.zeros(shots, dtype=bool)

    for i in range(time_steps):

        # Evaluate waveform at centre of each time interval
        t = (i + 0.5) * dt

        # Equal-prior optimal Dolinar magnitude
        ratio = 1.0 / (
            1.0 - np.exp(-4.0 * nbar * t)
        )

        # Physical/numerical displacement limit
        ratio = min(ratio, max_ratio)

        u_magnitude = alpha * np.sqrt(ratio)

        # Sign flips after every photon detection
        u = np.where(
            odd_parity,
            -u_magnitude,
            +u_magnitude
        )

        # Photon-counting intensity after displacement
        rate = np.abs(signal - u) ** 2

        # Probability of a photon detection during dt
        p_click = 1.0 - np.exp(-rate * dt)

        clicks = rng.random(shots) < p_click

        # Each detection flips the feedback sign
        odd_parity ^= clicks

    # Even number of detections -> guess +alpha
    guessed_plus = ~odd_parity

    error = np.mean(
        guessed_plus != sent_plus
    )

    return error

In [3]:
nbar_test = 0.5

dolinar_error = continuous_dolinar_error(
    nbar_test,
    shots=100_000,
    time_steps=1024,
    max_ratio=50,
    seed=1234
)

helstrom = helstrom_error(nbar_test)

print("Continuous Dolinar:", dolinar_error)
print("Helstrom bound:     ", helstrom)
print("Difference:         ", dolinar_error - helstrom)

Continuous Dolinar: 0.03666
Helstrom bound:      0.03506325248390313
Difference:          0.0015967475160968692
